# Eye Feature 재구성 — CASRN 조성벡터 → SMILES 기반 분자 descriptor

**배경**: 기존 eye 벡터는 CASRN을 컬럼으로 쓰는 조성벡터 방식이라, 학습에 없던 새 화합물은 해당 컬럼이 아예 없어서 feature 계산이 불가능함(연구목표 ② "새로운 화합물·조합까지 예측"과 충돌). 2026-10-06 팀 회의 결정에 따라 분자구조 기반으로 재구성함 — skin 쪽 `skin_feature_rebuild.ipynb` 와 같은 결정이고 같은 방법론임.

**방법**: SMILES → RDKit descriptor 계산. LogP·용해도·분자량 등 B(EPI Suite)로 얻으려던 물리화학 속성도 RDKit 자체 추정치로 전부 포함되므로, A/B를 나누지 않고 **RDKit 하나로 통합**.

**이 노트북이 하는 일**:
1. `eye_filtered.csv`(1,219행) 에서 성분별 SMILES 확보 — 결측은 PubChem 조회로 백필
2. 염/UVCB 를 **최대 프래그먼트**로 정리 (skin 과 다른 점, 아래 3단계에서 설명)
3. RDKit descriptor 계산 — skin 과 공통인 physchem 8개 + 전체 217개 + Morgan FP 2048
4. 제형 단위로 함량(%) 가중평균하여 feature 합성
5. 조성해시(`comp_group_id`) 부여 → `eye_vectorized.csv` 저장

> **skin 과의 관계**: 모델은 skin / eye 를 **따로** 만들기로 했으므로 feature 집합이 서로 달라도 됩니다. 다만 `MolWt`·`LogP`·`TPSA`·`HBD`·`HBA`·`RotatableBonds`·`AromaticProportion`·`ESOL_LogS` 8개는 **skin 과 똑같은 정의·똑같은 컬럼명**으로 계산해서, 나중에 두 endpoint 를 비교하거나 합칠 여지를 남겨둡니다.

> **배치 실행본**: 이 노트북과 동일한 계산을 하는 스크립트가 `build_vectors.py` 입니다. 노트북은 "왜 이렇게 했는지"를 설명하는 용도, 스크립트는 재실행용입니다. 두 결과가 어긋나지 않는지 아래 **QC 7단계**에서 실제로 대조합니다.

In [ ]:
import hashlib
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

RDLogger.DisableLog("rdApp.*")          # RDKit 경고 억제 (파싱 실패는 따로 집계함)

# 교내망이 SSL 인터셉션 프록시를 거치므로 OS 인증서 저장소를 써야 PubChem 이 열린다.
# 없으면 전부 CERTIFICATE_VERIFY_FAILED 로 실패한다 -> pip install truststore
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore 적용")
except ImportError:
    print("[warn] truststore 없음 - PubChem 조회가 SSL 오류로 실패할 수 있음")

BASE = Path(r"C:\Users\한세음\COS\2. vectorized\시도2")
SRC = Path(r"C:\Users\한세음\COS\1. filteration\eye_filtered.csv")
CACHE_PATH = BASE / "smiles_backfill.csv"

FP_BITS, FP_RADIUS = 2048, 2
print("작업 폴더:", BASE)

## 1단계 — 데이터 로드 및 제형 키 정의

`eye_filtered.csv` 는 **성분 1개 = 1행** 구조입니다(1,219행). 제형 단위로 묶어야 하는데 두 종류가 섞여 있습니다:

- **혼합물(Formulation)**: `formulation_id` 가 있음 (예: `EyeIrritation6pack_PID123`)
- **단일물질(Chemical)**: `formulation_id` 가 비어 있음 → `record_id` 로 "성분 1개짜리 제형" 취급

둘을 `form_key` 하나로 통일합니다.

In [ ]:
df = pd.read_csv(SRC)
df["form_key"] = df["formulation_id"].fillna("CHEM::" + df["record_id"].astype(str))

print(f"원본 {len(df)}행 / 제형 {df['form_key'].nunique()}개")
print()
print("substance_type 별:")
print(df.groupby("substance_type")["form_key"].nunique().to_string())
print()
print(f"SMILES 결측: {df['smiles'].isna().sum()}행 / CASRN 결측: {df['casrn'].isna().sum()}행")

## 2단계 — PubChem 으로 결측 SMILES 백필

SMILES 가 없는 64행을 PubChem PUG-REST 로 조회합니다. **CASRN → 화합물명** 순으로 시도합니다.

**주의사항 (실제로 겪은 것)**:
- **SSL 인터셉션**: 처음 돌렸을 때 19건 전부 실패했습니다. 오류가 `CERTIFICATE_VERIFY_FAILED: self-signed certificate in certificate chain` 인데, 교내망 프록시가 TLS 를 중간에서 다시 서명하기 때문입니다. `certifi` 번들에는 그 루트 인증서가 없어서 `truststore`(OS 인증서 저장소 사용)를 넣어야 열립니다.
- **UVCB 는 원래 안 나옵니다**: Polysorbate 20/80, Thyme oil, 지방산 칼륨염, benzylcoco 4급암모늄 등은 구조식이 하나로 정의되지 않는 물질군이라 "단일 SMILES" 자체가 존재하지 않습니다. 조회 실패가 정상입니다.

**캐시**: `smiles_backfill.csv` 가 있으면 재조회하지 않습니다. 같은 물질이 여러 행에 중복 등장하므로 **고유 (CASRN, 이름) 19건만** 조회합니다.

In [ ]:
def fetch_cid(query):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{urllib.parse.quote(str(query))}/cids/TXT"
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            txt = r.read().decode().strip()
        cid = txt.splitlines()[0].strip() if txt else ""
        return cid if cid and cid != "0" else None
    except Exception:
        return None


def fetch_smiles(cid):
    # 신규 PubChem 은 CanonicalSMILES 를 ConnectivitySMILES 로 옮겼다 -> 순차 시도
    for prop in ("CanonicalSMILES", "ConnectivitySMILES", "SMILES"):
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/{prop}/TXT"
            with urllib.request.urlopen(url, timeout=20) as r:
                txt = r.read().decode().strip()
            if txt:
                return txt.splitlines()[0].strip()
        except Exception:
            pass
        time.sleep(0.25)
    return None


miss = df[df["smiles"].isna()][["casrn", "chemical_name"]].drop_duplicates().reset_index(drop=True)
print(f"SMILES 결측 {int(df['smiles'].isna().sum())}행 -> 고유 조회대상 {len(miss)}건")

if CACHE_PATH.exists():
    cache = pd.read_csv(CACHE_PATH)
    print("기존 캐시 로드:", CACHE_PATH.name, f"({len(cache)}건)")
else:
    rows = []
    for i, r in miss.iterrows():
        cid = smi = None
        how = "none"
        for how_try, q in (("casrn", r["casrn"]), ("name", r["chemical_name"])):
            if not isinstance(q, str) or not q.strip():
                continue
            cid = fetch_cid(q)
            time.sleep(0.25)
            if cid:
                smi = fetch_smiles(cid)
                if smi:
                    how = how_try
                    break
        rows.append({"casrn": r["casrn"], "chemical_name": r["chemical_name"],
                     "smiles_new": smi, "cid": cid, "matched_by": how})
        print(f"  [{i+1:2d}/{len(miss)}] {'OK' if smi else '--'} {str(r['chemical_name'])[:50]}")
    cache = pd.DataFrame(rows)
    cache.to_csv(CACHE_PATH, index=False, encoding="utf-8-sig")

print()
print("조회 결과:", cache["matched_by"].value_counts().to_dict())

In [ ]:
# 백필 반영
df["smiles_source"] = np.where(df["smiles"].notna(), "original", "missing")
ok = cache[cache["smiles_new"].notna()][["casrn", "chemical_name", "smiles_new", "cid"]]
df = df.merge(ok, on=["casrn", "chemical_name"], how="left")

filled = df["smiles"].isna() & df["smiles_new"].notna()
df.loc[filled, "smiles"] = df.loc[filled, "smiles_new"]
df.loc[filled, "smiles_source"] = "pubchem"
df["pubchem_cid"] = np.where(filled, df["cid"], np.nan)
df = df.drop(columns=["smiles_new", "cid"])

print(f"백필된 행: {int(filled.sum())}")
print(f"SMILES 결측: 64 -> {int(df['smiles'].isna().sum())}행")
print()
print("백필 성공 물질:")
print(cache[cache["smiles_new"].notna()][["chemical_name", "matched_by"]].to_string(index=False))
print()
print("실패(= UVCB/무기물, 단일 SMILES 없음):")
print(cache[cache["smiles_new"].isna()]["chemical_name"].to_string(index=False))

## 3단계 — 염 / UVCB 를 최대 프래그먼트로 정리 ★ skin 과 다른 점

여기가 **skin 노트북에 없는 단계**이고, 실제로 값이 틀리는 걸 잡은 부분입니다.

SMILES 에 점(`.`)이 들어 있으면 분자가 둘 이상으로 쪼개져 있다는 뜻입니다. 두 경우가 섞여 있습니다:

| 유형 | 예시 | 문제 |
|---|---|---|
| **염** | `[Na+].[N-]1C=NC=N1` | Na⁺ 가 분자량에 더해짐 (보통 23~35 Da) |
| **UVCB 를 한 분자로 적은 것** | Abamectin = avermectin B1a + B1b | **두 성분의 MW·LogP 가 합산됨** |

RDKit 의 `Descriptors.MolWt` 등은 SMILES 전체를 하나의 분자로 보기 때문에, 보정하지 않으면 이렇게 나옵니다:

| 물질 | 보정 전 | 보정 후(최대 프래그먼트) | 실제 |
|---|---|---|---|
| Abamectin | MW 1732, LogP 10.4 | MW 873, LogP 5.4 | B1a 가 873 |
| Spinosad | MW 1478, LogP 11.4 | MW 746, LogP 5.9 | spinosyn A 가 732 |

**LogP 11** 은 실존 가능한 범위를 벗어난 값입니다. ESOL 공식이 `-0.63 × LogP` 항을 포함하므로 **ESOL 도 같이 틀어집니다** — skin 노트북 한계점 ③의 "아바멕틴 ESOL −16 극단치"가 바로 이 증상입니다(Delaney 공식의 한계만이 아니라 MW 가 2배로 들어간 탓).

그래서 descriptor 는 **최대 프래그먼트**(중원자 수 → 분자량 순) 기준으로 계산합니다. QSAR 에서 표준으로 쓰는 salt stripping 입니다. 제거된 질량은 `mw_removed` 로 남겨서 추적 가능하게 둡니다.

> **`sanitizeFrags=True` 가 필수입니다.** `False` 로 쪼개면 RDKit 내부 RingInfo 가 초기화되지 않아 `TPSA`·`FpDensityMorgan`·`SPS`·`NumAmideBonds` 가 **예외 없이 조용히 NaN** 으로 떨어집니다. 처음에 `False` 로 두는 실수를 했고 841제형 중 91개가 오염됐습니다.

> **Morgan FP 는 전체 분자 기준으로 둡니다.** 대응 이온(4급암모늄의 Cl⁻ 등)도 구조 정보로서 의미가 있고, FP 는 비트 단위라 염이 섞여도 MW 처럼 값이 왜곡되지 않습니다.

In [ ]:
def largest_fragment(m):
    """최대 프래그먼트(중원자 수 -> 분자량 순) 반환. sanitizeFrags=True 필수."""
    frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=True)
    if len(frags) == 1:
        return frags[0]
    return max(frags, key=lambda f: (f.GetNumHeavyAtoms(), Descriptors.MolWt(f)))


mols, mols_desc, canon = [], [], []
n_frags, mw_removed = [], []
for s in df["smiles"]:
    if pd.isna(s):
        mols.append(None); mols_desc.append(None); canon.append(np.nan)
        n_frags.append(np.nan); mw_removed.append(np.nan)
        continue
    m = Chem.MolFromSmiles(s)
    mols.append(m)
    canon.append(Chem.MolToSmiles(m) if m is not None else np.nan)
    if m is None:
        mols_desc.append(None); n_frags.append(np.nan); mw_removed.append(np.nan)
    else:
        lf = largest_fragment(m)
        mols_desc.append(lf)
        n_frags.append(len(Chem.GetMolFrags(m)))
        mw_removed.append(round(Descriptors.MolWt(m) - Descriptors.MolWt(lf), 4))

df["canonical_smiles"] = canon
df["parse_fail"] = [pd.notna(s) and m is None for s, m in zip(df["smiles"], mols)]
df["n_fragments"] = n_frags
df["mw_removed"] = mw_removed

print(f"다중 프래그먼트 행: {int((df['n_fragments'] > 1).sum())} / {int(df['n_fragments'].notna().sum())}")
print(f"  그중 50 Da 이상 제거: {int((df['mw_removed'] > 50).sum())}행")
print(f"RDKit 파싱 실패: {int(df['parse_fail'].sum())}행")

In [ ]:
# 보정 전후 비교 — 왜곡이 컸던 성분 상위
rows = []
for s, m, lf in zip(df["smiles"], mols, mols_desc):
    if m is None or lf is None or len(Chem.GetMolFrags(m)) < 2:
        continue
    rows.append({
        "MW_전체": round(Descriptors.MolWt(m), 1), "MW_보정": round(Descriptors.MolWt(lf), 1),
        "LogP_전체": round(Crippen.MolLogP(m), 2), "LogP_보정": round(Crippen.MolLogP(lf), 2),
    })
cmp_df = pd.DataFrame(rows).drop_duplicates()
cmp_df["LogP차"] = (cmp_df["LogP_전체"] - cmp_df["LogP_보정"]).round(2)
cmp_df = cmp_df.reindex(cmp_df["LogP차"].abs().sort_values(ascending=False).index)

name_map = df[["smiles", "chemical_name"]].drop_duplicates()
print("보정 효과가 가장 컸던 성분:")
print(cmp_df.head(10).to_string(index=False))

## 4단계 — RDKit descriptor 계산

세 묶음을 계산합니다.

**① skin 과 공통인 physchem 8개** — 컬럼명·정의를 skin 노트북과 똑같이 맞춥니다.

| descriptor | 의미 |
|---|---|
| `MolWt` | 분자량 |
| `LogP` | 지용성/수용성 정도 (Crippen 추정법) |
| `TPSA` | 극성 표면적 |
| `HBD` / `HBA` | 수소결합 공여/수용 개수 |
| `RotatableBonds` | 회전 가능 결합 수 (분자 유연성) |
| `AromaticProportion` | 방향족 원자 비율 |
| `ESOL_LogS` | 수용해도 추정치 (Delaney 2004 경험식) |

**② RDKit 전체 descriptor 217개** — `Descriptors._descList` 전체. 위 8개도 여기 포함되지만(`desc_` 접두로 구분), 구조 기반 정보를 넓게 쓰기 위해 전부 넣습니다. `Ipc` 처럼 값이 폭주하는 것은 `inf → NaN` 으로 정리합니다.

**③ Morgan fingerprint (ECFP4, r=2, 2048 bit)** — 부분구조 존재 여부. eye 에만 넣습니다.

ESOL 공식 (Delaney 2004):

$$\log S = 0.16 - 0.63\,\text{LogP} - 0.0062\,\text{MW} + 0.066\,\text{RotB} - 0.74\,\text{AP}$$

In [ ]:
DESC_COLS = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds",
             "AromaticProportion", "ESOL_LogS"]


def physchem_8(m):
    """skin 과 동일한 정의의 physchem 8개. m 은 최대 프래그먼트."""
    if m is None:
        return {c: np.nan for c in DESC_COLS}
    heavy = m.GetNumHeavyAtoms()
    arom = sum(1 for a in m.GetAtoms() if a.GetIsAromatic())
    ap = arom / heavy if heavy > 0 else 0.0
    mw = Descriptors.MolWt(m)
    logp = Crippen.MolLogP(m)
    rotb = Descriptors.NumRotatableBonds(m)
    return {
        "MolWt": mw, "LogP": logp, "TPSA": Descriptors.TPSA(m),
        "HBD": Descriptors.NumHDonors(m), "HBA": Descriptors.NumHAcceptors(m),
        "RotatableBonds": rotb, "AromaticProportion": ap,
        "ESOL_LogS": 0.16 - 0.63 * logp - 0.0062 * mw + 0.066 * rotb - 0.74 * ap,
    }


# 벡터화 가능한 행만 남긴다 (SMILES 결측 / 파싱 실패 제외)
keep = df["smiles"].notna() & ~df["parse_fail"]
dropped = df[~keep].copy()
dropped["drop_reason"] = np.where(dropped["smiles"].isna(), "smiles_missing", "rdkit_parse_fail")

comp = df[keep].reset_index(drop=True)
comp_mols = [m for m, k in zip(mols, keep) if k]
comp_mols_desc = [m for m, k in zip(mols_desc, keep) if k]

print(f"벡터화 대상 {len(comp)}행 (제외 {len(dropped)}행) / 고유 구조 {comp['canonical_smiles'].nunique()}종")

# ① physchem 8
p8 = pd.DataFrame([physchem_8(m) for m in comp_mols_desc])

# ② 전체 217 descriptor
desc_names = [n for n, _ in Descriptors._descList]
D = np.full((len(comp_mols_desc), len(desc_names)), np.nan)
for i, m in enumerate(comp_mols_desc):
    try:
        d = Descriptors.CalcMolDescriptors(m)
        D[i] = [d.get(n, np.nan) for n in desc_names]
    except Exception:
        pass
D[~np.isfinite(D)] = np.nan          # Ipc 등 폭주값 -> NaN

# ③ Morgan FP (전체 분자 기준)
gen = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)
FP = np.zeros((len(comp_mols), FP_BITS), dtype=np.float32)
for i, m in enumerate(comp_mols):
    FP[i] = np.asarray(gen.GetFingerprintAsNumPy(m), dtype=np.float32)

print(f"physchem {p8.shape[1]} + descriptor {len(desc_names)} + Morgan FP {FP_BITS}")
print()
print("physchem 8 요약:")
print(p8.describe().round(2).to_string())

## 5단계 — 제형 단위 함량(%) 가중평균

성분별 feature 를 `pct_active_ingredient` 가중평균으로 합쳐 제형 1행으로 만듭니다. edge case 2개를 같이 처리합니다:

1. **함량% 결측**: 가중치를 못 만들므로 **균등가중치**로 대체하고 `weight_mode="equal_fallback"` 로 표시합니다. 단일성분 제형이면 결과에 영향이 없고(평균 대상이 하나), 다성분이면 "조성비를 모른다"는 정보를 컬럼으로 남기는 셈입니다.
2. **descriptor 가 NaN 인 성분**: 무기물·금속(NaOH, ZnCl₂, Sb₂O₃ 등)은 Gasteiger 전하와 `BCUT2D` 가 원래 정의되지 않습니다. 이런 칸은 **NaN 을 빼고 남은 성분으로만** 가중평균합니다(NaN 하나가 제형 전체를 NaN 으로 만들지 않도록).

In [ ]:
comp["w_raw"] = pd.to_numeric(comp["pct_active_ingredient"], errors="coerce")

weights = np.empty(len(comp))
weight_mode = {}
for key, idx in comp.groupby("form_key").groups.items():
    idx = np.asarray(idx)
    w = comp.loc[idx, "w_raw"].to_numpy(dtype=float)
    if np.isfinite(w).all() and np.nansum(w) > 0:
        w, mode = w / w.sum(), "pct_active_ingredient"
    else:
        w, mode = np.full(len(idx), 1.0 / len(idx)), "equal_fallback"
    weight_mode[key] = mode
    weights[comp.index.get_indexer(idx)] = w
comp["weight"] = weights

print("가중치 방식:", pd.Series(weight_mode).value_counts().to_dict())


def wavg(mat, w):
    """NaN 무시 가중평균. 전부 NaN 인 열은 NaN 유지."""
    mask = np.isfinite(mat)
    denom = (mask * w[:, None]).sum(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom > 0,
                        np.nansum(np.where(mask, mat, 0) * w[:, None], axis=0) / denom,
                        np.nan)


form_keys = comp["form_key"].drop_duplicates().tolist()
pos = {k: i for i, k in enumerate(form_keys)}

P8 = np.full((len(form_keys), len(DESC_COLS)), np.nan)
DD = np.full((len(form_keys), len(desc_names)), np.nan)
FF = np.zeros((len(form_keys), FP_BITS), dtype=np.float32)
p8_np = p8[DESC_COLS].to_numpy(dtype=float)
meta_rows = []

for key, g in comp.groupby("form_key", sort=False):
    r = pos[key]
    loc = comp.index.get_indexer(np.asarray(g.index))
    w = comp["weight"].to_numpy()[loc]

    P8[r] = wavg(p8_np[loc], w)
    DD[r] = wavg(D[loc], w)
    FF[r] = (FP[loc] * w[:, None]).sum(axis=0)

    epa = g["epa_class"].dropna().unique()
    meta_rows.append({
        "form_key": key,
        "source": g["source"].iloc[0],
        "substance_type": g["substance_type"].iloc[0],
        "formulation_name": g["formulation_name"].iloc[0],
        "comp_count": len(g),
        "weight_mode": weight_mode[key],
        "pct_sum_used": g["w_raw"].sum(min_count=1),
        "n_smiles_from_pubchem": int(g["smiles_source"].eq("pubchem").sum()),
        "component_casrns": " | ".join(g["casrn"].fillna("NA").astype(str)),
        "component_names": " | ".join(g["chemical_name"].fillna("NA").astype(str)),
        "final_label": epa[0] if len(epa) else np.nan,
        "internal_conflict_resolved": len(epa) > 1,
        "ghs_class": (g["ghs_class"].dropna().unique()[:1] or [np.nan])[0],
        "draize_score": g["draize_score"].dropna().mean() if g["draize_score"].notna().any() else np.nan,
    })

meta = pd.DataFrame(meta_rows)
meta["is_single_component"] = meta["comp_count"].eq(1)
meta["final_label"] = pd.to_numeric(meta["final_label"], errors="coerce")
meta["binary_label"] = np.where(meta["final_label"].isin([1, 2]), 1,
                         np.where(meta["final_label"].isin([3, 4]), 0, np.nan))

print(f"제형 {len(meta)}개로 집계")
print("라벨 분포:", meta["final_label"].value_counts(dropna=False).sort_index().to_dict())

## 6단계 — 조성해시 `comp_group_id` 와 조성 완전성

**왜 필요한가**: 같은 조성 제형이 여러 행에 들어 있습니다. 이게 train 과 test 로 갈리면 모델이 사실상 답을 외운 상태로 평가돼서 **성능이 실제보다 높게 나옵니다.** 그룹키로 묶어서 같은 조성은 한쪽 fold 에만 보내야 합니다.

```python
StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0).split(X, y, groups=df["comp_group_id"])
```

**규칙**: `"CAS:%"` 를 정렬해 이어붙인 뒤 md5 앞 10자 (3주차 `build_harmonized.py` 와 동일). SMILES 유무와 무관하게 **원본 전체 행**에서 계산해서, 구조를 못 구한 UVCB 성분도 조성 비교에는 반영되게 합니다.

> skin 신규 노트북은 그룹키를 대표 `MIX` id 로 쓰는데, **분할 결과는 이 해시와 동일**합니다(ARI = 1.0 로 확인). ID 문자열만 다르고 묶는 기능은 같으니, endpoint 별로 따로 학습하는 한 문제되지 않습니다.

**조성 완전성**: FIFRA 신고서는 활성성분만 적고 계면활성제·용매 같은 inert 는 안 적습니다. 그래서 함량 합이 100% 가 안 됩니다. 재정규화하지 않고 세 컬럼으로 분해해서 남깁니다 — AD(적용범위) 신뢰도 축으로 쓸 수 있습니다.

In [ ]:
ck = df["casrn"].where(df["casrn"].notna() & df["casrn"].astype(str).ne(""), np.nan)
ckf = pd.DataFrame({
    "form_key": df["form_key"], "ck": ck.fillna("__UNK__"), "is_unknown": ck.isna(),
    "pct": pd.to_numeric(df["pct_active_ingredient"], errors="coerce"),
})
ckf["pct_missing"] = ckf["pct"].isna()
ckf["pct"] = ckf["pct"].fillna(100.0)                 # 원액 취급 (해시 계산용에만)
ckf = ckf.drop_duplicates(["form_key", "ck", "pct"])
ckf["ing_key"] = ckf["ck"] + ":" + ckf["pct"].round(6).map(lambda x: f"{x:g}")

gk = ckf.groupby("form_key")
grp = pd.DataFrame({
    "composition_key": gk["ing_key"].apply(lambda s: "|".join(sorted(s))),
    "unknown_count": gk["is_unknown"].sum().astype(int),
    "known_pct": gk.apply(lambda g: g.loc[~g["is_unknown"], "pct"].sum(), include_groups=False).round(6),
    "unknown_pct": gk.apply(lambda g: g.loc[g["is_unknown"], "pct"].sum(), include_groups=False).round(6),
})
grp["comp_group_id"] = grp["composition_key"].map(lambda k: "C" + hashlib.md5(k.encode()).hexdigest()[:10])
grp["undeclared_pct"] = (100.0 - grp["known_pct"] - grp["unknown_pct"]).clip(lower=0).round(6)

meta = meta.merge(grp.reset_index(), on="form_key", how="left")
meta["group_size"] = meta["comp_group_id"].map(meta["comp_group_id"].value_counts()).astype(int)
meta["is_composition_twin"] = meta["group_size"].gt(1)
_c = meta.groupby("comp_group_id")["final_label"].nunique()
meta["group_label_conflict"] = meta["comp_group_id"].map(_c > 1)

# 구조를 못 구한 성분이 있는 제형은 구조벡터가 조성 전체를 대표하지 못한다
n_declared = df.groupby("form_key").size()
meta["n_components_declared"] = meta["form_key"].map(n_declared).astype(int)
meta["n_components_dropped"] = meta["n_components_declared"] - meta["comp_count"]
meta["structure_coverage"] = (meta["comp_count"] / meta["n_components_declared"]).round(6)

print(f"제형 {len(meta)} / 고유 조성그룹 {meta['comp_group_id'].nunique()}")
print(f"  쌍둥이 있는 제형 {int(meta['is_composition_twin'].sum())}  "
      f"그중 라벨충돌 {int(meta[meta['is_composition_twin']]['group_label_conflict'].sum())}")
print(f"  성분 일부가 구조 미해결인 제형 {int(meta['n_components_dropped'].gt(0).sum())}")
print()
print("조성 완전성:")
print(meta[["known_pct", "unknown_pct", "undeclared_pct", "structure_coverage"]].describe().round(1).to_string())

## 7단계 — 저장 및 `build_vectors.py` 와 대조 (QC)

평면 CSV 로 저장합니다. 그리고 **같은 계산을 하는 배치 스크립트 `build_vectors.py` 의 결과와 실제로 일치하는지** 대조합니다. 노트북과 스크립트가 따로 굴러가면서 조용히 어긋나는 걸 막는 장치입니다.

In [ ]:
X = pd.concat([
    meta[["form_key", "source", "substance_type", "formulation_name", "comp_count",
          "is_single_component", "weight_mode", "pct_sum_used", "final_label", "binary_label",
          "internal_conflict_resolved", "ghs_class", "draize_score",
          "comp_group_id", "group_size", "is_composition_twin", "group_label_conflict",
          "known_pct", "unknown_pct", "undeclared_pct", "unknown_count",
          "n_components_declared", "n_components_dropped", "structure_coverage",
          "n_smiles_from_pubchem", "component_casrns", "component_names", "composition_key"]]
        .rename(columns={"form_key": "formulation_id"}),
    pd.DataFrame(P8, columns=DESC_COLS).round(6),
    pd.DataFrame(FF, columns=[f"fp_w_{i}" for i in range(FP_BITS)]).round(6),
    pd.DataFrame(DD, columns=[f"desc_{n}" for n in desc_names]).round(6),
], axis=1)
X.insert(1, "endpoint", "eye")
X["final_label"] = X["final_label"].astype("Int64")
X["binary_label"] = X["binary_label"].astype("Int64")

X.to_csv(BASE / "eye_vectorized_nb.csv", index=False, encoding="utf-8-sig")
print(f"eye_vectorized_nb.csv  {X.shape[0]}행 x {X.shape[1]}컬럼")
print(f"  physchem {len(DESC_COLS)} + Morgan FP {FP_BITS} + descriptor {len(desc_names)}")

In [ ]:
# build_vectors.py 결과와 대조
ref_path = BASE / "eye_vectorized.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    print("제형 수 일치?", len(ref) == len(X), f"({len(ref)} vs {len(X)})")
    same_ids = set(ref["formulation_id"]) == set(X["formulation_id"])
    print("제형 ID 집합 일치?", same_ids)
    print("comp_group_id 일치?", set(ref["comp_group_id"]) == set(X["comp_group_id"]))

    a = ref.set_index("formulation_id").sort_index()
    b = X.set_index("formulation_id").sort_index()
    print()
    print("공통 descriptor 열 최대 절대차:")
    for c in ["desc_MolWt", "desc_MolLogP", "desc_TPSA", "desc_NumHDonors", "fp_w_37"]:
        if c in a.columns and c in b.columns:
            diff = (a[c] - b[c]).abs().max()
            print(f"  {c:22s} {diff:.2e}")
    # 노트북의 physchem 8 은 build_vectors.py 의 desc_ 와 같은 값이어야 한다
    pairs = [("MolWt", "desc_MolWt"), ("LogP", "desc_MolLogP"), ("TPSA", "desc_TPSA"),
             ("HBD", "desc_NumHDonors"), ("HBA", "desc_NumHAcceptors"),
             ("RotatableBonds", "desc_NumRotatableBonds"), ("ESOL_LogS", "aux_ESOL_logS")]
    print()
    print("physchem 8 vs build_vectors.py 대응열 최대 절대차:")
    for nb_c, sc_c in pairs:
        if sc_c in a.columns:
            print(f"  {nb_c:20s} vs {sc_c:24s} {(a[sc_c] - b[nb_c]).abs().max():.2e}")
else:
    print("[warn] eye_vectorized.csv 없음 - build_vectors.py 를 먼저 실행하면 대조됩니다")

In [ ]:
# 제외된 행 확인
print(f"제외 {len(dropped)}행 (= 구조를 끝까지 못 구한 성분)")
print(dropped["drop_reason"].value_counts().to_dict())
print()
print(dropped.groupby("chemical_name").size().sort_values(ascending=False).to_string())
print()
fully_lost = set(df["form_key"]) - set(meta["form_key"])
print(f"제형 전체가 소실된 경우: {len(fully_lost)}개")
lost = df[df["form_key"].isin(fully_lost)]
print("  라벨 분포:", lost["epa_class"].value_counts(dropna=False).to_dict())

## 결과 시각화 — feature 가 실제로 어떻게 나왔는지 보기

지금까지는 숫자(개수·%)로만 확인했는데, 여기서부터는 **feature 값 자체가 어떤 분포로 나왔는지, 자극/비자극 사이에 차이가 보이는지**를 그림으로 확인합니다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_font_path = r"C:\Windows\Fonts\malgun.ttf"
fm.fontManager.addfont(_font_path)
plt.rcParams["font.family"] = fm.FontProperties(fname=_font_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

SURFACE = "#fcfcfb"; INK = "#0b0b0b"; MUTED = "#898781"; GRID_C = "#e1e0d9"; BASE_C = "#c3c2b7"
GOOD = "#0ca30c"; CRIT = "#d03b3b"; BLUE = "#2a78d6"
%matplotlib inline

### ① SMILES 커버리지 — 백필이 얼마나 메웠나

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.0), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
n_total = len(df)
n_orig = int((df["smiles_source"] == "original").sum())
n_bf = int((df["smiles_source"] == "pubchem").sum())
n_none = n_total - n_orig - n_bf

ax.barh([0], [n_total], color="#e8e7e2", edgecolor=BASE_C, height=0.5, zorder=2)
ax.barh([0], [n_orig + n_bf], color="#cdeee1", edgecolor=GOOD, height=0.5, zorder=3)
ax.barh([0], [n_orig], color="#a9ddcb", edgecolor=GOOD, height=0.5, zorder=4)
ax.text(n_orig / 2, 0, f"원본 {n_orig}행", ha="center", va="center", fontsize=9.5,
        fontweight="bold", color=INK)
ax.text(n_orig + n_bf / 2, 0, f"+{n_bf}", ha="center", va="center", fontsize=9, color=INK)
ax.text(n_total + 8, 0, f"{n_none}행\n미해결(UVCB)", ha="left", va="center", fontsize=8.5, color=MUTED)
ax.set_xlim(0, n_total + 130); ax.set_ylim(-0.5, 0.5)
ax.set_yticks([]); ax.tick_params(labelsize=8.5, colors=MUTED)
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(BASE_C)
ax.set_title(f"성분 행 {n_total}개의 SMILES 확보 현황 — PubChem 백필로 {n_bf}행 추가",
             fontsize=11, color=INK, fontweight="bold", loc="left", pad=10)
plt.tight_layout(); plt.show()

### ② 염 보정 효과 — 보정하지 않으면 LogP 가 얼마나 틀리나

In [ ]:
top = cmp_df.head(12)
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
y = np.arange(len(top))
ax.hlines(y, top["LogP_보정"], top["LogP_전체"], color=BASE_C, linewidth=1.6, zorder=2)
ax.scatter(top["LogP_보정"], y, s=55, color=GOOD, zorder=3, label="보정 후 (최대 프래그먼트)")
ax.scatter(top["LogP_전체"], y, s=55, color=CRIT, zorder=3, label="보정 전 (SMILES 전체)")
ax.axvspan(-3, 6, color="#f2f1ec", zorder=1)
ax.text(6.1, len(top) - 0.6, "← 통상적 LogP 범위", fontsize=8.5, color=MUTED)
ax.set_yticks(y); ax.set_yticklabels([f"MW {int(v)}" for v in top["MW_전체"]], fontsize=8.5)
ax.set_xlabel("Crippen LogP", fontsize=9.5, color=MUTED)
ax.tick_params(labelsize=8.5, colors=MUTED)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.spines["left"].set_color(BASE_C); ax.spines["bottom"].set_color(BASE_C)
ax.grid(axis="x", color=GRID_C, linewidth=0.7, zorder=0)
ax.legend(fontsize=8.5, frameon=False, loc="lower right")
ax.set_title("염/UVCB 보정 전후 LogP — 보정 없이는 실존 범위를 벗어난 값이 들어간다",
             fontsize=11, color=INK, fontweight="bold", loc="left", pad=10)
plt.tight_layout(); plt.show()

### ③ physchem 8개 분포 — 성분이 어떤 값 범위에 퍼져 있나

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), facecolor=SURFACE)
for ax, col in zip(axes.flat, DESC_COLS):
    ax.set_facecolor(SURFACE)
    ax.hist(p8[col].dropna(), bins=25, color=BLUE, alpha=0.85, edgecolor="white", linewidth=0.3)
    ax.set_title(col, fontsize=10.5, color=INK, fontweight="bold")
    ax.tick_params(labelsize=8, colors=MUTED)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(BASE_C); ax.spines["bottom"].set_color(BASE_C)
    ax.grid(axis="y", color=GRID_C, linewidth=0.7, zorder=0)
fig.suptitle(f"성분 {len(p8)}행 physchem 분포 (제형 가중평균 이전, 염 보정 후)",
             fontsize=12, color=INK, fontweight="bold", x=0.01, ha="left")
plt.tight_layout(); plt.show()

### ④ descriptor 간 상관관계 — 서로 겹치는 정보는 없는지

In [ ]:
corr = p8[DESC_COLS].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5), facecolor=SURFACE)
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(DESC_COLS))); ax.set_xticklabels(DESC_COLS, rotation=45, ha="right", fontsize=8.5)
ax.set_yticks(range(len(DESC_COLS))); ax.set_yticklabels(DESC_COLS, fontsize=8.5)
for i in range(len(DESC_COLS)):
    for j in range(len(DESC_COLS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else INK)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("physchem 8개 상관행렬", fontsize=11, color=INK, fontweight="bold", loc="left", pad=10)
plt.tight_layout(); plt.show()

hi = [(DESC_COLS[i], DESC_COLS[j], round(corr.iloc[i, j], 3))
      for i in range(len(DESC_COLS)) for j in range(i + 1, len(DESC_COLS))
      if abs(corr.iloc[i, j]) >= 0.9]
print("|r| >= 0.9 쌍:", hi if hi else "없음")

**관찰**: `ESOL_LogS` 와 `LogP` 가 강한 음의 상관을 보입니다 — ESOL 공식 자체가 `-0.63 × LogP` 항을 포함하니 당연한 결과지만, **둘 다 feature 로 같이 넣으면 중복정보**라는 뜻입니다. Kang2023 벤치마크 논문도 Pearson `|r| ≥ 0.9` 기준으로 다중공선성을 제거했으니, 모델링 단계에서 둘 중 하나를 빼거나 PCA 로 묶는 걸 검토해야 합니다. skin 노트북에서도 같은 관찰(-0.93)이 나왔습니다.

### ⑤ 자극(1) vs 비자극(0) — feature 가 라벨과 관련 있어 보이는지 1차 확인

모델을 돌리기 전에, 제형 단위로 합성한 feature 가 라벨과 **눈으로 봐도 차이가 나는지** 먼저 봅니다. 차이가 안 보여도 모델이 못 배운다는 뜻은 아니지만(비선형 조합으로 배울 수 있음), 보이면 최소한 '헛짓은 아니다'는 심증은 얻습니다.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5), facecolor=SURFACE)
plot_df = X.dropna(subset=["binary_label"]).copy()
for ax, col in zip(axes.flat, DESC_COLS):
    ax.set_facecolor(SURFACE)
    d0 = plot_df.loc[plot_df["binary_label"] == 0, col].dropna()
    d1 = plot_df.loc[plot_df["binary_label"] == 1, col].dropna()
    bp = ax.boxplot([d0, d1], tick_labels=["비자극(0)", "자극(1)"], patch_artist=True,
                    widths=0.55, medianprops=dict(color=INK, linewidth=1.5), showfliers=False)
    bp["boxes"][0].set_facecolor("#cdeee1"); bp["boxes"][0].set_edgecolor(GOOD)
    bp["boxes"][1].set_facecolor("#f6d4d2"); bp["boxes"][1].set_edgecolor(CRIT)
    ax.set_title(f"{col}\n중앙값 {d0.median():.1f} vs {d1.median():.1f}",
                 fontsize=10, color=INK, fontweight="bold")
    ax.tick_params(labelsize=8, colors=MUTED)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(BASE_C); ax.spines["bottom"].set_color(BASE_C)
    ax.grid(axis="y", color=GRID_C, linewidth=0.7, zorder=0)
fig.suptitle(f"제형 단위 feature × 이진라벨  (비자극 {int((plot_df['binary_label']==0).sum())} / "
             f"자극 {int((plot_df['binary_label']==1).sum())})",
             fontsize=12, color=INK, fontweight="bold", x=0.01, ha="left")
plt.tight_layout(); plt.show()

### ⑥ 조성 완전성 — FIFRA 미기재 비율이 얼마나 큰가

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), facecolor=SURFACE)
ax = axes[0]; ax.set_facecolor(SURFACE)
ax.hist(meta["undeclared_pct"], bins=30, color="#e0a23a", alpha=0.9, edgecolor="white", linewidth=0.3)
ax.axvline(meta["undeclared_pct"].median(), color=CRIT, linestyle="--", linewidth=1.4)
ax.text(meta["undeclared_pct"].median() + 2, ax.get_ylim()[1] * 0.9,
        f"중앙값 {meta['undeclared_pct'].median():.1f}%", fontsize=9, color=CRIT)
ax.set_xlabel("undeclared_pct (FIFRA 미기재 inert 비율)", fontsize=9.5, color=MUTED)
ax.set_ylabel("제형 수", fontsize=9.5, color=MUTED)
ax.set_title("조성이 얼마나 비어 있나", fontsize=11, color=INK, fontweight="bold", loc="left")

ax = axes[1]; ax.set_facecolor(SURFACE)
gs = meta["group_size"].value_counts().sort_index()
ax.bar(gs.index.astype(str), gs.values, color=BLUE, alpha=0.85, edgecolor="white")
for x, v in zip(gs.index.astype(str), gs.values):
    ax.text(x, v, str(v), ha="center", va="bottom", fontsize=8.5, color=INK)
ax.set_xlabel("같은 조성 그룹의 크기", fontsize=9.5, color=MUTED)
ax.set_ylabel("제형 수", fontsize=9.5, color=MUTED)
ax.set_title("조성 중복 — 그룹 분할이 필요한 이유", fontsize=11, color=INK, fontweight="bold", loc="left")

for ax in axes:
    ax.tick_params(labelsize=8.5, colors=MUTED)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(BASE_C); ax.spines["bottom"].set_color(BASE_C)
    ax.grid(axis="y", color=GRID_C, linewidth=0.7, zorder=0)
plt.tight_layout(); plt.show()

## 한계점 (팀 공유 필요)

1. **SMILES 미해결 22행 → 제형 8개가 완전 소실**: Polysorbate 20/80, Thyme oil, 지방산 칼륨염, benzylcoco 4급암모늄 등은 **구조식이 하나로 정의되지 않는 UVCB** 라 PubChem 에도 단일 SMILES 가 없습니다. "분자 하나의 구조"라는 전제가 안 맞는 물질이라 RDKit 방식의 **근본적 한계**입니다. 조성의 일부만 구조화된 제형은 `structure_coverage < 1` 로 표시해 뒀습니다.

2. **UVCB 를 최대 프래그먼트로 대표한 것은 근사입니다**: Abamectin(B1a+B1b), Spinosad(spinosyn A+D), Pyrethrins 는 실제로 여러 성분의 혼합물인데 가장 큰 성분 하나로 대표했습니다. 합산값을 쓰는 것보다는 훨씬 타당하지만 실제 혼합물은 아닙니다. `aux_mw_removed` / `n_fragments` 로 어떤 행이 그런지 추적할 수 있습니다.

3. **무기물·금속은 일부 descriptor 가 원천적으로 NaN**: NaOH, CaO, KOH, ZnCl₂, Sb₂O₃, Cu₂O, Maneb, triphenyltin 등 10개 제형에서 `BCUT2D_*` 와 `MaxPartialCharge` 계열이 NaN 입니다. Gasteiger 전하 파라미터가 해당 원소에 정의되어 있지 않기 때문이고, 계산 실패가 아니라 **정의되지 않음**입니다. 모델링 시 결측 처리 방침을 정해야 합니다.

4. **ESOL 정확도**: 문헌 실측 logS 와 대조했을 때 RMSE 약 1.3 log unit 이었습니다(Delaney 원논문 보고값 ~0.9보다 나쁨). 특히 Hexane 은 +2.0 어긋납니다 — 소분자·고용해도 영역에서 ESOL 이 약한 건 알려진 한계입니다. **절대값으로 쓰지 말고 순서 정보(feature)로만** 쓰는 게 안전합니다.

5. **`ESOL_LogS` ↔ `LogP` 다중공선성**: 공식 자체가 LogP 를 포함해서 생기는 구조적 중복입니다. 모델링 전에 하나를 빼거나 묶어야 합니다.

6. **라벨 결측 47제형**: GHS 만 있고 EPA 4단계가 없는 제형입니다. `has_label` 로 구분되고 학습에서는 제외됩니다.

7. **같은 조성인데 라벨이 다른 그룹 24제형**: 동일 조성이 다른 Cat 으로 보고된 경우입니다. 원본 데이터의 불일치이므로 최악값을 쓸지, 제외할지 팀에서 정해야 합니다. `group_label_conflict` 로 표시해 뒀습니다.

8. **함량% 결측 → 균등가중치 대체**: `weight_mode == "equal_fallback"` 인 제형이 상당수입니다. 단일성분은 영향이 없지만 다성분이면 "조성비를 모르는 채 평균한 것"이므로 가중평균 feature 의 신뢰도가 낮습니다.

## 팀에 공유할 파일

### 반드시 공유 (3개)

| 파일 | 내용 | 왜 필요한가 |
|---|---|---|
| `eye_vectorized.csv` | **841행 × 2304컬럼** — 최종 feature + 라벨 + 메타 | 모델링 입력. 이 파일 하나로 학습 가능 |
| `eye_feature_rebuild.ipynb` | 이 노트북 | 각 숫자가 왜 그렇게 나왔는지의 근거. 한계점 포함 |
| `build_vectors.py` | 배치 재실행 스크립트 | 원본이 갱신되면 이걸 돌려서 재생성 |

### 재현에 필요 (3개)

| 파일 | 내용 |
|---|---|
| `smiles_backfill.csv` | PubChem 조회 캐시 19건 — 있으면 네트워크 재조회 안 함 |
| `component_level_clean.csv` | 성분 단위 중간 산출물 (SMILES 출처·염 제거량·ESOL 추적용) |
| `dropped_rows.csv` | 구조를 못 구해 제외된 23행 + 이유 |

### 선택 (모델링 방식에 따라)

| 파일 | 내용 |
|---|---|
| `eye_vectorized_multionly.csv` | 228행 — 다성분 제형만 (혼합물 전용 모델 쓸 때) |
| `X_combined_labeled.parquet` / `y_labeled.parquet` | 라벨 있는 794행만, parquet (CSV 로딩이 느릴 때) |
| `X_morgan_any.parquet` | FP 를 가중평균 대신 OR 집계한 버전 (대안 표현) |
| `vectorization_log.json` | 전 단계 집계 수치 (보고서 작성용) |

### 공유하지 않아도 되는 것

- `vectorization_log_prev.json`, `build_vectors_prev.py.bak` — 개정 이전 버전 백업 (염 보정·ESOL 추가 전)
- `X_descriptors.parquet`, `X_morgan_weighted.parquet` — `eye_vectorized.csv` 에 이미 합쳐져 있음
- `eye_vectorized_nb.csv` — 이 노트북이 QC 대조용으로 만든 사본. `eye_vectorized.csv` 와 같은 내용

---

### 모델링 담당자에게 전달할 내용 3가지

**1. 분할은 반드시 그룹 기준으로** — 같은 조성이 train/test 로 갈리면 성능이 부풀려집니다.

```python
from sklearn.model_selection import StratifiedGroupKFold

d = pd.read_csv("eye_vectorized.csv")
d = d[d["final_label"].notna()]                      # 794행
feat = [c for c in d.columns if c.startswith(("fp_w_", "desc_"))] + DESC_COLS
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
splits = list(cv.split(d[feat], d["binary_label"], groups=d["comp_group_id"]))
```

**2. `ESOL_LogS` 와 `LogP` 중 하나는 빼야 합니다** (상관 −0.9 이상).

**3. NaN 처리 방침이 필요합니다** — 무기물 10제형의 `BCUT2D_*` 계열은 대치(imputation)보다 **해당 열을 쓰지 않거나 결측 표시자를 함께 넣는** 쪽이 맞습니다. 값이 없는 게 아니라 정의되지 않는 것이기 때문입니다.

### skin 쪽에 전달할 사항 1가지

skin 노트북 3단계 `compute_descriptors` 에 **염 제거가 빠져 있습니다.** LogP 가 1단위 이상 틀리는 성분이 19종, 영향받는 제형이 54개(그중 단일성분 20개 — 가중평균 희석이 없어 왜곡이 그대로 반영)입니다. Spinosad LogP 11.4, Abamectin 10.4 처럼 실존 범위를 벗어난 값이 들어가 있고, ESOL 도 같이 틀어집니다. skin 노트북 한계점 ③에 적힌 "아바멕틴 ESOL −16 극단치"가 이 증상입니다.

고치는 건 3줄입니다:

```python
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)   # True 필수
    if len(frags) > 1:
        mol = max(frags, key=lambda f: (f.GetNumHeavyAtoms(), Descriptors.MolWt(f)))
    ...
```

SMILES 캐시를 그대로 쓰니 재조회 없이 몇 초면 끝나고, 제형 수와 라벨 분포는 바뀌지 않습니다.